# Preprocesamiento de las fuentes — pandas y NumPy

Este notebook prepara **contratos, convocatorias, entidades contratantes, RNP y sancionados** por separado. Tú ejecutas las celdas, revisas los DataFrames y decides cuándo guardar cada Excel usando el botón al final de su sección. No hace joins, no concatena años ni construye el target.

## 1. Estructura y ejecución local o en Colab

```text
proyecto/
├── notebooks/preprocesamiento.ipynb
└── data/
    ├── raw/
    │   ├── contratos/CONOSCE_CONTRATOS2023_0.xlsx ...
    │   ├── convocatorias/CONOSCE_CONVOCATORIAS2023_0.xlsx ...
    │   ├── entidades_contratantes/entidades_contratantes.csv
    │   ├── rnp/Reporte_Sancionados_Tribunal_....xls
    │   └── sancionados/sancionados.csv y sancionados_multa.csv
    └── processed/  ← aquí guardarán los botones
```

**Jupyter local:** abre el notebook desde `proyecto/notebooks` o desde la raíz del proyecto. La configuración busca `data/raw` en el directorio de trabajo y sus padres.

**Google Colab:** abrir el notebook no carga automáticamente la carpeta `data`. Cada persona debe subir/descomprimir el proyecto, clonarlo desde su repositorio o montar Drive. Después usa una ruta relativa hacia el proyecto en `PROYECTO_RELATIVO`, por ejemplo `Path("G5-IA-APLICADA")` si está junto al directorio de trabajo, o cambia el directorio de trabajo a la raíz del proyecto. No hay ninguna ruta de Windows escrita en el código.

Si se usa Drive, el montaje pertenece a cada usuario. Desde el directorio de trabajo de Colab puedes configurar una ruta relativa como `Path("drive/MyDrive/G5-IA-APLICADA")` **solo si esa es la ubicación real en tu Drive**. OneDrive no se monta automáticamente en Colab.

**Salidas:** al hacer clic se guarda el Excel en `proyecto/data/processed`. En Colab también se inicia su descarga al navegador; la carpeta de descarga en tu laptop depende de tu navegador. Para conservarlo entre sesiones de Colab, guarda el proyecto en Drive o descarga el Excel.

Se detectan los archivos existentes, incluidos 2023–2026. El año 2026 puede estar incompleto; el año del nombre es una cohorte, no un filtro sobre las fechas de cada registro.

In [ ]:
# Si falta alguna dependencia, descomenta y ejecuta esta línea:
# %pip install "pandas>=2.0" numpy openpyxl lxml ipywidgets xlsxwriter
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from lxml import html
from IPython.display import display, FileLink
import ipywidgets as widgets

try:
    from google.colab import files as colab_files
    EN_COLAB = True
except ImportError:
    colab_files = None
    EN_COLAB = False

print("pandas:", pd.__version__, "| NumPy:", np.__version__, "| Colab:", EN_COLAB)


In [ ]:
# Si el proyecto está en otro directorio, escribe su ubicación RELATIVA aquí.
# Ejemplos: Path("G5-IA-APLICADA") o Path("../G5-IA-APLICADA").
# Path(".") busca desde el directorio de trabajo actual hacia sus padres.
PROYECTO_RELATIVO = Path(".")
ANIOS = None  # None = todos los archivos. Ejemplo: [2023, 2024, 2025].
ELIMINAR_DUPLICADOS_EXACTOS = True

def localizar_proyecto(ruta_relativa):
    if ruta_relativa.is_absolute():
        raise ValueError("Configura una ruta relativa, no absoluta.")
    inicio = Path.cwd() / ruta_relativa
    candidatos = [inicio] if ruta_relativa != Path(".") else [inicio, *inicio.parents]
    for candidato in candidatos:
        if (candidato / "data" / "raw").is_dir():
            return candidato
    raise FileNotFoundError(
        "No se encontró data/raw. Sube o clona el proyecto y ajusta PROYECTO_RELATIVO. "
        f"Directorio de trabajo actual: {Path.cwd()}"
    )

ROOT = localizar_proyecto(PROYECTO_RELATIVO)
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
for carpeta in ["contratos", "convocatorias", "entidades_contratantes", "rnp", "sancionados"]:
    assert (RAW / carpeta).is_dir(), f"Falta la carpeta data/raw/{carpeta}"

print("Directorio de trabajo:", Path.cwd())
print("Proyecto detectado:", ROOT)
print("Entradas:", RAW)
print("Los botones guardarán en:", PROCESSED)
# Todavía no se crea ni se guarda ningún Excel.


## 2. Reglas comunes y botón de salida

- Encabezados en minúsculas, sin tildes y con guiones bajos.
- Recortar espacios y unificar vacíos explícitos.
- Mantener identificadores como texto y conservar ceros iniciales; generar una copia sin ceros únicamente para código de entidad.
- Conservar fechas y montos originales, añadiendo versiones `_dt` y `_num`.
- Eliminar solo duplicados exactos si la configuración lo indica. Marcar claves repetidas sin seleccionar arbitrariamente una versión.
- Conservar nulos, RUC dudosos, montos negativos y fechas incoherentes con indicadores de revisión. No imputar nulos como cero.

El formato del RUC no acredita su existencia ante SUNAT. No se sustituyen RUC de contratista por destinatario de pago ni se convierten monedas. Escalado, imputación y codificación para modelos quedan para una etapa posterior.

El botón **no guarda nada al ejecutar su celda**: primero aparece y tú debes hacer clic. En cada Excel, los años o tablas se guardan en hojas distintas, sin unirlos. La hoja `_resumen` describe las entradas y la limpieza.

In [ ]:
def nombre_columna(nombre):
    s = unicodedata.normalize("NFKD", str(nombre))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def texto_limpio(serie):
    s = serie.astype("string").str.strip()
    return s.mask(s.str.upper().isin(["", "NULL", "NONE", "NAN", "N/A"]))

def preparar_base(df, archivo):
    df = df.copy()
    columnas = [nombre_columna(c) for c in df.columns]
    if len(columnas) != len(set(columnas)):
        raise ValueError("Encabezados duplicados después de normalizar: revisar manualmente.")
    df.columns = columnas
    for c in df.columns:
        df[c] = texto_limpio(df[c])
    filas_entrada = len(df)
    repetidos = df.duplicated(keep="first")
    n_repetidos = int(repetidos.sum())
    # Número de fila aproximado para Excel/CSV con encabezado en fila 1.
    df["fila_origen"] = np.arange(2, len(df) + 2)
    df["archivo_origen"] = archivo.name
    df["ruta_origen"] = archivo.relative_to(RAW).as_posix()
    if ELIMINAR_DUPLICADOS_EXACTOS:
        df = df.loc[~repetidos].copy()
    return df, {"filas_entrada": filas_entrada,
                "duplicados_exactos_detectados": n_repetidos,
                "duplicados_exactos_eliminados": n_repetidos if ELIMINAR_DUPLICADOS_EXACTOS else 0}

def exigir_columnas(df, columnas):
    faltantes = sorted(set(columnas) - set(df.columns))
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}. Revisa el archivo antes de continuar.")

def normalizar_id(serie):
    # Corrige la terminación .0 causada por representaciones numéricas de Excel.
    return texto_limpio(serie).str.replace(r"\.0+$", "", regex=True)

def agregar_id(df, columna, quitar_ceros=False):
    if columna not in df:
        return
    s = normalizar_id(df[columna])
    if quitar_ceros:
        s = s.str.lstrip("0").replace("", "0")
    df[f"{columna}_normalizado"] = s

def agregar_ruc(df, columna):
    if columna not in df:
        return
    agregar_id(df, columna)
    s = df[f"{columna}_normalizado"]
    df[f"{columna}_formato_11_digitos"] = s.str.fullmatch(r"\d{11}").fillna(False)
    df[f"{columna}_prefijo_usual"] = s.str.fullmatch(r"(?:10|15|17|20)\d{9}").fillna(False)
    df[f"{columna}_requiere_revision"] = ~df[f"{columna}_formato_11_digitos"]

def convertir_fecha(serie, formato=None):
    s = texto_limpio(serie)
    if formato is not None:
        return pd.to_datetime(s, format=formato, errors="coerce")
    salida = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    # Algunos archivos contienen fechas como YYYYMMDD.
    compacto = s.str.fullmatch(r"\d{8}").fillna(False)
    salida.loc[compacto] = pd.to_datetime(s.loc[compacto], format="%Y%m%d", errors="coerce")
    # Otros guardan fechas como números seriales de Excel.
    n = pd.to_numeric(s, errors="coerce")
    serial = (~compacto & n.between(20000, 80000)).fillna(False)
    salida.loc[serial] = pd.Timestamp("1899-12-30") + pd.to_timedelta(n.loc[serial], unit="D")
    resto = s.notna() & ~compacto & ~serial
    salida.loc[resto] = pd.to_datetime(s.loc[resto], format="mixed", dayfirst=True, errors="coerce")
    return salida

def agregar_fecha(df, columna, formato=None):
    if columna not in df:
        return
    salida = convertir_fecha(df[columna], formato)
    df[f"{columna}_dt"] = salida
    df[f"{columna}_error_parseo"] = df[columna].notna() & salida.isna()

def agregar_monto(df, columna):
    if columna not in df:
        return
    # Los archivos revisados usan punto decimal. No asumir el significado de una coma.
    s = texto_limpio(df[columna])
    s = s.str.replace(r"^(?:S/\.?|PEN)\s*", "", regex=True, case=False)
    valor = pd.to_numeric(s, errors="coerce")
    no_finito = valor.notna() & ~np.isfinite(valor.astype(float))
    valor = valor.mask(no_finito)
    df[f"{columna}_num"] = valor
    df[f"{columna}_error_parseo"] = df[columna].notna() & valor.isna()
    df[f"{columna}_negativo"] = valor.lt(0).fillna(False)

def agregar_moneda(df):
    if "moneda" not in df:
        return
    s = texto_limpio(df["moneda"]).str.upper()
    equivalencias = {"SOLES": "PEN", "SOL": "PEN", "NUEVOS SOLES": "PEN",
                     "NUEVO SOL": "PEN", "PEN": "PEN", "DOLARES": "USD",
                     "DÓLARES": "USD", "DOLAR AMERICANO": "USD", "USD": "USD"}
    df["moneda_normalizada"] = s.replace(equivalencias)
    df["es_moneda_pen"] = df["moneda_normalizada"].eq("PEN").fillna(False)

def marcar_clave(df, clave):
    exigir_columnas(df, clave)
    completos = df[clave].notna().all(axis=1)
    df["clave_incompleta"] = ~completos
    df["clave_repetida"] = completos & df.duplicated(clave, keep=False)

def revisar(df):
    display(df.head(3))
    display(pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(),
                          "porcentaje_nulos": (df.isna().mean() * 100).round(2)}))
    indicadores = [c for c in df if (c.endswith(("_error_parseo", "_negativo", "_requiere_revision"))
                                    or c.startswith("flag_") or c in ["clave_incompleta", "clave_repetida"])]
    if indicadores:
        display(df[indicadores].sum().rename("filas_marcadas").to_frame())


In [ ]:
def anio_archivo(archivo):
    encontrado = re.search(r"(?:CONTRATOS|CONVOCATORIAS)(20\d{2})", archivo.stem, flags=re.I)
    if encontrado is None:
        raise ValueError(f"No se identificó el año en {archivo.name}")
    return int(encontrado.group(1))

def listar_anuales(fuente):
    archivos = sorted((RAW / fuente).rglob("*.xlsx"))
    if ANIOS is not None:
        archivos = [p for p in archivos if anio_archivo(p) in ANIOS]
    if not archivos:
        raise FileNotFoundError(f"No hay Excel seleccionados en data/raw/{fuente}")
    return archivos

def boton_excel(nombre_excel, tablas, resumen):
    if not tablas:
        raise ValueError("Primero ejecuta el preprocesamiento de esta sección.")
    boton = widgets.Button(
        description="Guardar y descargar Excel" if EN_COLAB else "Guardar Excel en processed",
        button_style="success", layout=widgets.Layout(width="290px")
    )
    salida = widgets.Output()

    def guardar(_):
        boton.disabled = True
        with salida:
            salida.clear_output()
            try:
                # Verificar límites antes de crear el archivo.
                nombres_hojas = set()
                for nombre, df in tablas.items():
                    if len(df) > 1048575 or len(df.columns) > 16384:
                        raise ValueError(f"La tabla {nombre} supera el límite de una hoja Excel.")
                    hoja = re.sub(r"[\[\]:*?/\\]", "_", nombre)[:31]
                    if not hoja or hoja in nombres_hojas or hoja == "_resumen":
                        raise ValueError("Nombres de hojas duplicados o inválidos.")
                    nombres_hojas.add(hoja)
                PROCESSED.mkdir(parents=True, exist_ok=True)
                destino = PROCESSED / nombre_excel
                # Texto como texto: evita convertir identificadores y descripciones en fórmulas/URLs.
                with pd.ExcelWriter(destino, engine="xlsxwriter", date_format="yyyy-mm-dd",
                                    datetime_format="yyyy-mm-dd hh:mm:ss",
                                    engine_kwargs={"options": {"strings_to_formulas": False,
                                                              "strings_to_urls": False}}) as writer:
                    for nombre, df in tablas.items():
                        hoja = re.sub(r"[\[\]:*?/\\]", "_", nombre)[:31]
                        df.to_excel(writer, sheet_name=hoja, index=False)
                        writer.sheets[hoja].freeze_panes(1, 0)
                        writer.sheets[hoja].set_column(0, len(df.columns) - 1, 20)
                    pd.DataFrame(resumen).to_excel(writer, sheet_name="_resumen", index=False)
                print("Excel guardado:", destino)
                if EN_COLAB:
                    colab_files.download(str(destino))
                else:
                    # En Jupyter el enlace se sirve desde el directorio del notebook.
                    import os
                    display(FileLink(os.path.relpath(destino, start=Path.cwd())))
            except Exception as exc:
                print("No se completó la salida:", exc)
            finally:
                boton.disabled = False

    boton.on_click(guardar)
    display(boton, salida)
    print("Haz clic cuando termines de revisar. Repetir el clic reemplaza solo este Excel.")
    return boton

def resumen_archivo(archivo, df, stats, tabla):
    return {"tabla": tabla, "archivo_entrada": archivo.relative_to(RAW).as_posix(),
            **stats, "filas_salida": len(df), "columnas_salida": len(df.columns)}


## 3. Contratos

Limpiar los identificadores, convertir fechas y montos y normalizar moneda. La clave a revisar es entidad–contrato–ítem–contratista. Las modificaciones contractuales y fechas finales actualizadas se conservan, pero su disponibilidad histórica debe revisarse antes de usarlas como predictores. No sumar el monto total por ítem: puede estar repetido.

**Resultado en memoria:** `dfs_contratos`, un diccionario de DataFrames por archivo. **Salida manual:** `data/processed/contratos_preprocesados.xlsx`.

In [ ]:
def preprocesar_contratos(df, archivo):
    df, stats = preparar_base(df, archivo)
    exigir_columnas(df, ["codigoentidad", "codigoconvocatoria", "n_cod_contrato", "num_item",
                          "ruc_contratista", "fecha_publicacion_contrato", "fecha_suscripcion_contrato"])
    for c in ["codigoentidad", "codigoconvocatoria", "n_cod_contrato", "codigo_contrato", "num_item"]:
        agregar_id(df, c, quitar_ceros=(c == "codigoentidad"))
    for c in ["ruc_contratista", "ruc_destinatario_pago"]:
        agregar_ruc(df, c)
    for c in ["fecha_publicacion_contrato", "fecha_suscripcion_contrato", "fecha_vigencia_inicial",
              "fecha_vigencia_final", "fecha_vigencia_fin_actualizada"]:
        agregar_fecha(df, c)
    for c in ["monto_contratado_total", "monto_contratado_item", "monto_adicional",
              "monto_reduccion", "monto_prorroga", "monto_complementario"]:
        agregar_monto(df, c)
    agregar_moneda(df)
    marcar_clave(df, ["codigoentidad_normalizado", "n_cod_contrato_normalizado",
                      "num_item_normalizado", "ruc_contratista_normalizado"])
    df["flag_publicacion_antes_suscripcion"] = (df["fecha_publicacion_contrato_dt"] < df["fecha_suscripcion_contrato_dt"])
    if {"fecha_vigencia_inicial_dt", "fecha_vigencia_final_dt"}.issubset(df.columns):
        df["flag_fin_antes_inicio"] = df["fecha_vigencia_final_dt"] < df["fecha_vigencia_inicial_dt"]
    for c in ["tieneresolucion", "espublicado"]:
        if c in df:
            df[f"{c}_normalizado"] = df[c].str.upper()
    df["anio_cohorte_archivo"] = anio_archivo(archivo)
    return df, stats


In [ ]:
archivos_contratos = listar_anuales("contratos")
display(pd.DataFrame({"archivo": [p.relative_to(RAW).as_posix() for p in archivos_contratos],
                      "anio": [anio_archivo(p) for p in archivos_contratos]}))

dfs_contratos = {}
resumen_contratos = []
for archivo in archivos_contratos:
    print("Leyendo:", archivo.name)
    entrada = pd.read_excel(archivo, sheet_name=0, dtype="string", engine="openpyxl", keep_default_na=False)
    df, stats = preprocesar_contratos(entrada, archivo)
    dfs_contratos[archivo.stem] = df
    resumen_contratos.append(resumen_archivo(archivo, df, stats, archivo.stem))
    print("Filas limpias:", len(df))
    del entrada
# Cambia la clave si quieres revisar otro archivo.
revisar(dfs_contratos[next(iter(dfs_contratos))])


In [ ]:
# SALIDA DE CONTRATOS: DataFrames disponibles y botón para obtener el Excel.
display(pd.DataFrame(resumen_contratos))
boton_contratos = boton_excel("contratos_preprocesados.xlsx", dfs_contratos, resumen_contratos)


## 4. Convocatorias

Unificar sector/nivel de gobierno, identificadores, categorías, fechas y montos referenciales. Se conserva `COL_10` cuando existe y se crea el encabezado común. Revisar su contenido antes de utilizarlo. Marcar claves entidad–convocatoria–ítem repetidas y fechas anteriores a la convocatoria.

**Resultado:** `dfs_convocatorias`. **Salida manual:** `data/processed/convocatorias_preprocesadas.xlsx`.

In [ ]:
def preprocesar_convocatorias(df, archivo):
    df, stats = preparar_base(df, archivo)
    exigir_columnas(df, ["codigoentidad", "codigoconvocatoria", "n_item", "entidad_ruc", "fecha_convocatoria"])
    if "sector_nivel_de_gobierno" in df:
        df["sector_nivel_gobierno"] = df["sector_nivel_de_gobierno"]
    elif "col_10" in df:
        df["sector_nivel_gobierno"] = df["col_10"]
    else:
        raise ValueError("No se encontró el encabezado esperado de sector/nivel de gobierno.")
    for c in ["codigoentidad", "codigoconvocatoria", "n_item", "codigoitem"]:
        agregar_id(df, c, quitar_ceros=(c == "codigoentidad"))
    agregar_ruc(df, "entidad_ruc")
    for c in ["fecha_convocatoria", "fechaintegracionbases", "fechapresentacionpropuesta"]:
        agregar_fecha(df, c)
    for c in ["montoreferencial", "monto_referencial_item"]:
        agregar_monto(df, c)
    agregar_moneda(df)
    for c in ["objetocontractual", "tipoprocesoseleccion", "sector_nivel_gobierno", "tipoentidad",
              "departamento_item", "provincia_item", "distrito_item", "paquete"]:
        if c in df:
            df[f"{c}_normalizado"] = df[c].str.upper()
    marcar_clave(df, ["codigoentidad_normalizado", "codigoconvocatoria_normalizado", "n_item_normalizado"])
    for c in ["fechaintegracionbases", "fechapresentacionpropuesta"]:
        if f"{c}_dt" in df:
            df[f"flag_{c}_antes_convocatoria"] = df[f"{c}_dt"] < df["fecha_convocatoria_dt"]
    df["anio_cohorte_archivo"] = anio_archivo(archivo)
    return df, stats


In [ ]:
archivos_convocatorias = listar_anuales("convocatorias")
dfs_convocatorias = {}
resumen_convocatorias = []
for archivo in archivos_convocatorias:
    print("Leyendo:", archivo.name)
    entrada = pd.read_excel(archivo, sheet_name=0, dtype="string", engine="openpyxl", keep_default_na=False)
    df, stats = preprocesar_convocatorias(entrada, archivo)
    dfs_convocatorias[archivo.stem] = df
    resumen_convocatorias.append(resumen_archivo(archivo, df, stats, archivo.stem))
    print("Filas limpias:", len(df))
    del entrada
revisar(dfs_convocatorias[next(iter(dfs_convocatorias))])


In [ ]:
# SALIDA DE CONVOCATORIAS.
display(pd.DataFrame(resumen_convocatorias))
boton_convocatorias = boton_excel("convocatorias_preprocesadas.xlsx", dfs_convocatorias, resumen_convocatorias)


## 5. Entidades contratantes

La carpeta correcta es `entidades_contratantes`. Leer el CSV con `|` y `cp1252`; normalizar RUC y códigos, convertir última actualización y marcar códigos CONSUCODE vinculados a varios RUC. No eliminar entidades inactivas.

**Resultado:** `df_entidades`. **Salida manual:** `data/processed/entidades_contratantes_preprocesadas.xlsx`.

In [ ]:
def preprocesar_entidades(df, archivo):
    df, stats = preparar_base(df, archivo)
    exigir_columnas(df, ["ruc", "codconsucode", "nombre_de_entidad", "ultimaactualizacion"])
    agregar_ruc(df, "ruc")
    agregar_id(df, "codconsucode", quitar_ceros=True)
    agregar_id(df, "codigo_siaf")
    agregar_fecha(df, "ultimaactualizacion", formato="%d/%m/%y")
    for c in ["nombre_de_entidad", "departamento", "provincia", "distrito", "estado"]:
        if c in df:
            df[f"{c}_normalizado"] = df[c].str.upper()
    marcar_clave(df, ["codconsucode_normalizado"])
    # Control interno de esta fuente; no es un join con otra tabla.
    diferentes = df.groupby("codconsucode_normalizado")["ruc_normalizado"].transform("nunique")
    df["flag_codigo_con_varios_ruc"] = diferentes.gt(1).fillna(False)
    return df, stats


In [ ]:
archivo_entidades = RAW / "entidades_contratantes" / "entidades_contratantes.csv"
entrada = pd.read_csv(archivo_entidades, sep="|", encoding="cp1252", dtype="string", keep_default_na=False)
df_entidades, stats_entidades = preprocesar_entidades(entrada, archivo_entidades)
revisar(df_entidades)


In [ ]:
# SALIDA DE ENTIDADES CONTRATANTES.
display(df_entidades.head())
boton_entidades = boton_excel(
    "entidades_contratantes_preprocesadas.xlsx", {"entidades": df_entidades},
    [resumen_archivo(archivo_entidades, df_entidades, stats_entidades, "entidades")]
)


## 6. RNP

El `.xls` oficial contiene HTML tabular; se lee con lxml y se convierte a DataFrames. Mantener inhabilitaciones y multas separadas, conservar vigentes y no vigentes, convertir fechas y marcar episodios ambiguos. Un fin vacío puede corresponder a una sanción definitiva: no imputarlo.

Un inicio más de un año después del año de la resolución se marca para revisión; no demuestra una reanudación. No se construye el target.

**Resultados:** `df_rnp_inhabilitaciones` y `df_rnp_multas`. **Salida manual:** `data/processed/rnp_preprocesado.xlsx`, con dos hojas de datos.

In [ ]:
COLUMNAS_RNP_INH = ["numero", "razon_social", "ruc_proveedor", "resolucion", "periodo",
                    "desde_original", "hasta_original", "infraccion", "otra_infraccion", "norma", "estado_actual"]
COLUMNAS_RNP_MULTA = ["numero", "razon_social", "ruc_proveedor", "resolucion", "fecha_resolucion_original",
                      "monto_multa", "infraccion", "periodo_suspension", "desde_original", "hasta_original",
                      "otra_infraccion", "norma", "verificacion_pago", "estado_actual"]

def leer_rnp(archivo):
    documento = html.fromstring(archivo.read_bytes().decode("cp1252"))
    filas_inh, filas_multa = [], []
    for fila in documento.xpath("//tr"):
        celdas = fila.xpath("./td|./th")
        # Evita encabezados y tablas contenedoras anidadas de la exportación.
        if len(celdas) not in (11, 14) or not celdas[0].text_content().strip().isdigit():
            continue
        valores = [" ".join(c.text_content().split()) for c in celdas]
        (filas_inh if len(celdas) == 11 else filas_multa).append(valores)
    if not filas_inh or not filas_multa:
        raise ValueError("No se reconocieron ambas tablas. Revisar la estructura del XLS del RNP.")
    inhabilitaciones = pd.DataFrame(filas_inh, columns=COLUMNAS_RNP_INH)
    multas = pd.DataFrame(filas_multa, columns=COLUMNAS_RNP_MULTA)
    for nombre, tabla in [("inhabilitaciones", inhabilitaciones), ("multas", multas)]:
        numeros = pd.to_numeric(tabla["numero"], errors="raise").to_numpy(dtype=int)
        if not np.array_equal(np.sort(numeros), np.arange(1, len(tabla) + 1)):
            raise ValueError(f"Numeración incompleta o duplicada en {nombre}; revisar la exportación.")
        print(nombre, "| filas leídas:", len(tabla))
    return inhabilitaciones, multas

def preprocesar_rnp(df, archivo, tipo):
    df, stats = preparar_base(df, archivo)
    # En RNP fila_origen identifica posición dentro de la tabla extraída, no fila de Excel.
    agregar_ruc(df, "ruc_proveedor")
    for c in ["desde_original", "hasta_original", "fecha_resolucion_original"]:
        agregar_fecha(df, c)
    df["estado_actual_normalizado"] = df["estado_actual"].str.upper()
    df["resolucion_normalizada"] = df["resolucion"].str.upper()
    df["tipo_tabla"] = tipo
    df["anio_resolucion"] = pd.to_numeric(df["resolucion"].str.extract(r"-(\d{4})(?:-|$)")[0], errors="coerce").astype("Int64")
    df["flag_fin_antes_inicio"] = df["hasta_original_dt"] < df["desde_original_dt"]
    if tipo == "inhabilitacion":
        periodo = df["periodo"].str.upper()
        df["tipo_inhabilitacion"] = np.select(
            [periodo.eq("DEFINITIVO").fillna(False).to_numpy(dtype=bool), periodo.notna().to_numpy(dtype=bool)],
            ["definitiva", "temporal"], default="sin_periodo")
        df["flag_inicio_posterior_anio_resolucion"] = (df["desde_original_dt"].dt.year > df["anio_resolucion"] + 1).fillna(False)
        df["flag_inicio_faltante"] = df["desde_original_dt"].isna()
    else:
        agregar_monto(df, "monto_multa")
        df["flag_fecha_resolucion_faltante"] = df["fecha_resolucion_original_dt"].isna()
    marcar_clave(df, ["ruc_proveedor_normalizado", "resolucion_normalizada", "desde_original_dt"])
    n_fechas = df.groupby(["ruc_proveedor_normalizado", "resolucion_normalizada"])["desde_original_dt"].transform("nunique")
    df["flag_resolucion_varios_inicios"] = n_fechas.gt(1).fillna(False)
    return df, stats


In [ ]:
archivos_rnp = sorted((RAW / "rnp").glob("*.xls"))
if not archivos_rnp:
    raise FileNotFoundError("No se encontró la exportación XLS en data/raw/rnp.")
display(pd.DataFrame({"archivo": [p.name for p in archivos_rnp]}))
# Si hay varias exportaciones, selecciona explícitamente un índice: 0, 1, ...
INDICE_RNP = 0
archivo_rnp = archivos_rnp[INDICE_RNP]
print("Exportación seleccionada:", archivo_rnp.name)
entrada_inh, entrada_multas = leer_rnp(archivo_rnp)
df_rnp_inhabilitaciones, stats_rnp_inh = preprocesar_rnp(entrada_inh, archivo_rnp, "inhabilitacion")
df_rnp_multas, stats_rnp_multas = preprocesar_rnp(entrada_multas, archivo_rnp, "multa")
revisar(df_rnp_inhabilitaciones)
revisar(df_rnp_multas)


In [ ]:
# SALIDA DEL RNP: dos DataFrames y un único Excel con dos hojas.
display(df_rnp_inhabilitaciones.head())
display(df_rnp_multas.head())
boton_rnp = boton_excel(
    "rnp_preprocesado.xlsx",
    {"inhabilitaciones": df_rnp_inhabilitaciones, "multas": df_rnp_multas},
    [resumen_archivo(archivo_rnp, df_rnp_inhabilitaciones, stats_rnp_inh, "inhabilitaciones"),
     resumen_archivo(archivo_rnp, df_rnp_multas, stats_rnp_multas, "multas")]
)


## 7. Sancionados CONOSCE

Leer cada CSV con `|` y `cp1252`; mantener motivos de infracción como texto y convertir fechas `YYYYMMDD`. Conservar fecha de corte de la fuente, ambos tipos de sanción y todas las filas con indicadores de calidad. El monto se convierte únicamente en multas.

**Resultados:** `df_sancionados` y `df_sancionados_multas`. **Salida manual:** `data/processed/sancionados_preprocesados.xlsx`, con dos hojas de datos.

Después de ejecutar la última celda, puedes inspeccionar cualquiera de esos DataFrames. Haz clic en el botón para guardar el Excel y, en Colab, descargarlo. Las fuentes siguen separadas del RNP.

In [ ]:
def preprocesar_sancionados(df, archivo, tipo):
    df, stats = preparar_base(df, archivo)
    exigir_columnas(df, ["ruc", "fecha_corte", "fecha_inicio", "fecha_fin", "numero_resolucion"])
    agregar_ruc(df, "ruc")
    for c in ["fecha_corte", "fecha_inicio", "fecha_fin"]:
        agregar_fecha(df, c, formato="%Y%m%d")
    df["numero_resolucion_normalizado"] = df["numero_resolucion"].str.upper()
    df["tipo_tabla"] = tipo
    df["flag_fin_antes_inicio"] = df["fecha_fin_dt"] < df["fecha_inicio_dt"]
    df["flag_inicio_despues_corte_fuente"] = df["fecha_inicio_dt"] > df["fecha_corte_dt"]
    df["flag_inicio_faltante"] = df["fecha_inicio_dt"].isna()
    if tipo == "multa":
        exigir_columnas(df, ["monto"])
        agregar_monto(df, "monto")
    marcar_clave(df, ["ruc_normalizado", "numero_resolucion_normalizado", "fecha_inicio_dt"])
    return df, stats


In [ ]:
archivo_sancionados = RAW / "sancionados" / "sancionados.csv"
archivo_sancionados_multas = RAW / "sancionados" / "sancionados_multa.csv"
entrada_inh = pd.read_csv(archivo_sancionados, sep="|", encoding="cp1252", dtype="string", keep_default_na=False)
entrada_multas = pd.read_csv(archivo_sancionados_multas, sep="|", encoding="cp1252", dtype="string", keep_default_na=False)
df_sancionados, stats_sancionados = preprocesar_sancionados(entrada_inh, archivo_sancionados, "inhabilitacion")
df_sancionados_multas, stats_sancionados_multas = preprocesar_sancionados(entrada_multas, archivo_sancionados_multas, "multa")
revisar(df_sancionados)
revisar(df_sancionados_multas)


In [ ]:
# SALIDA FINAL DE SANCIONADOS: DataFrames y botón de Excel.
display(df_sancionados.head())
display(df_sancionados_multas.head())
boton_sancionados = boton_excel(
    "sancionados_preprocesados.xlsx",
    {"inhabilitaciones": df_sancionados, "multas": df_sancionados_multas},
    [resumen_archivo(archivo_sancionados, df_sancionados, stats_sancionados, "inhabilitaciones"),
     resumen_archivo(archivo_sancionados_multas, df_sancionados_multas, stats_sancionados_multas, "multas")]
)
